In [1]:
%load_ext autoreload
%autoreload 2

import sys
sys.path.insert(0, "../src")

In [2]:
import plotly.express as px

In [5]:
from data.load import process
from data.timeseries import build_series

In [6]:
!ls ../data/raw

202512-llegadas-de-turistas-extranjeros.xlsx


In [7]:
RAW_FILE = '../data/raw/202512-llegadas-de-turistas-extranjeros.xlsx'
PROCESSED_FILE = '../data/processed/llegadas-de-turistas-extranjeros.parquet'

df = process(RAW_FILE, PROCESSED_FILE)

# Explora

In [8]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 186777 entries, 0 to 186776
Data columns (total 6 columns):
 #   Column           Non-Null Count   Dtype         
---  ------           --------------   -----         
 0   period           186777 non-null  datetime64[us]
 1   region           186777 non-null  str           
 2   country          186777 non-null  str           
 3   crossing_type    186777 non-null  str           
 4   crossing_detail  186777 non-null  str           
 5   arrivals         186777 non-null  int64         
dtypes: datetime64[us](1), int64(1), str(4)
memory usage: 18.0 MB


In [9]:
df.head()

,period,region,country,crossing_type,crossing_detail,arrivals
0,2013-01-01,Europa,Alemania,Aeropuertos,Concordia (Chacalluta),41
1,2013-01-01,America Del Sur,Argentina,Aeropuertos,Concordia (Chacalluta),264
2,2013-01-01,Oceania,Australia,Aeropuertos,Concordia (Chacalluta),77
3,2013-01-01,Europa,Austria,Aeropuertos,Concordia (Chacalluta),3
4,2013-01-01,Europa,Belgica,Aeropuertos,Concordia (Chacalluta),3


In [10]:
df.groupby("crossing_detail")["arrivals"].sum().to_clipboard()

In [11]:
top5_country = df.groupby("country")["arrivals"].sum().nlargest(5)
top5_crossing_type = df.groupby("crossing_type")["arrivals"].sum().nlargest(5)
top5_crossing_detail = df.groupby("crossing_detail")["arrivals"].sum().nlargest(5)

fig1 = px.bar(top5_country, orientation="h", title="Top 5 países por llegadas totales")
fig1.show()

fig2 = px.bar(top5_crossing_type, orientation="h", title="Top 5 tipos de paso")
fig2.show()

fig3 = px.bar(top5_crossing_detail, orientation="h", title="Top 5 pasos fronterizos")
fig3.show()

# Series de tiempo

In [16]:
import pandas as pd 

df = pd.read_parquet(PROCESSED_FILE)

In [17]:
import plotly.graph_objects as go

fig1 = go.Figure()
for crossing in top5_crossing_type.index:
    ts = build_series(df, crossing_type=crossing)
    fig1.add_trace(go.Scatter(x=ts.index, y=ts.values, name=crossing))
fig1.update_layout(title="Llegadas mensuales por tipo de paso")
fig1.show()

fig2 = go.Figure()
for detail in top5_crossing_detail.index:
    ts = build_series(df, crossing_detail=detail)
    fig2.add_trace(go.Scatter(x=ts.index, y=ts.values, name=detail))
fig2.update_layout(title="Llegadas mensuales por paso fronterizo")
fig2.show()

In [13]:
from plotly.subplots import make_subplots

fig = make_subplots(rows=3, cols=2, subplot_titles=list(top5_country.index))

for i, country in enumerate(top5_country.index):
    row = i // 2 + 1
    col = i % 2 + 1
    ts = build_series(df, country=country)
    fig.add_trace(go.Scatter(x=ts.index, y=ts.values, name=country, showlegend=False), row=row, col=col)

fig.update_layout(height=900, title="Llegadas mensuales — Top 5 países")
fig.show()

# Desc seasonal

In [14]:
components = ["trend", "seasonal", "resid"]
crossings = list(top5_crossing_type.index)

fig = make_subplots(
    rows=3, cols=3,
    subplot_titles=crossings * 3,
    row_titles=["Trend", "Seasonal", "Residual"],
)

for col_i, crossing in enumerate(crossings):
    ts = build_series(df, crossing_type=crossing).replace(0, 1)
    stl = STL(ts, period=13).fit()
    for row_i, comp in enumerate(components):
        values = getattr(stl, comp)
        fig.add_trace(
            go.Scatter(x=values.index, y=values.values, showlegend=False),
            row=row_i + 1, col=col_i + 1,
        )

fig.update_layout(height=900, title="STL Decomposition — Top 5 tipos de paso")
fig.show()

NameError: name 'STL' is not defined

In [ ]:
from statsmodels.tsa.seasonal import STL
from plotly.subplots import make_subplots

components = ["trend", "seasonal", "resid"]
countries = list(top5_country.index)

fig = make_subplots(
    rows=3, cols=5,
    subplot_titles=countries * 3,
    row_titles=["Trend", "Seasonal", "Residual"],
)

for col_i, country in enumerate(countries):
    ts = build_series(df, country=country).replace(0, 1)
    stl = STL(ts, period=13).fit()
    for row_i, comp in enumerate(components):
        values = getattr(stl, comp)
        fig.add_trace(
            go.Scatter(x=values.index, y=values.values, showlegend=False),
            row=row_i + 1, col=col_i + 1,
        )

fig.update_layout(height=900, title="STL Decomposition — Top 5 países")
fig.show()